## 2026 EY AI & Data Challenge - Landsat Data Extraction Notebook

This notebook demonstrates Landsat data extraction and the creation of an output file to be used by the benchmark notebook. The baseline data is [Landsat Collection 2 Level 2](https://planetarycomputer.microsoft.com/dataset/landsat-c2-l2) data from the MS Planetary Computer catalog.

## Environment preparation

In [ ]:
!pip install uv
!uv pip install  -r ../requirements.txt 

In [1]:
import warnings
warnings.filterwarnings('ignore')

# Data manipulation and analysis
import numpy as np
import pandas as pd

# Process catalog data
from odc.stac import stac_load
from pystac.extensions.eo import EOExtension as eo
import planetary_computer as pc

from datetime import date
from tqdm import tqdm
import os

# Multithreading
from concurrent.futures import ThreadPoolExecutor

# Retry
import time
from pystac_client.exceptions import APIError

# Importing functions
import sys
import os
sys.path.append(os.path.abspath('..'))
from utils import save_df, get_catalog, landsat_bands_of_interest

In [ ]:
tqdm.pandas()

In [ ]:
error_row = pd.Series(
    {band: np.nan for band in landsat_bands_of_interest}
)

# Path
save_path = 'snow://workspace/USER$.PUBLIC."waterscript-ey-data-science-challenge"/versions/live/data/intermediate/'

# We want a ~100 m buffer around each point.  
# At the equator, 1 degree ≈ 110 km. 
# Therefore, the degree equivalent of 100 m is: buffer_deg ≈ 100 m / 110,000 m per degree ≈ 0.00089831
# This value ensures that the buffer approximately matches the pixel resolution of Landsat imagery, capturing
# a ~100 m area around each sampling location.
bbox_size = 0.00089831
bbox_for_operation = bbox_size / 2

# MS Planetary Catalog Collection
catalog = get_catalog(
    'https://planetarycomputer.microsoft.com/api/stac/v1'
)

In [ ]:
# Df for training
water_quality_df = (
    pd.read_csv('../data/raw/water_quality_training_dataset.csv')
)

# Df for testing
validation_df = (
    pd.read_csv('../data/raw/submission_template.csv')
)

### Functions

In [ ]:
def get_bbox(row):
    '''
    Calculates a ~100m bounding box centered on the sampling coordinates.
    '''
    lat = row['Latitude']
    lon = row['Longitude']
    
    bbox = [
        lon - bbox_for_operation,
        lat - bbox_for_operation,
        lon + bbox_for_operation,
        lat + bbox_for_operation
    ]

    return bbox

In [ ]:
def get_items(row, cloud_cover_query):
    '''
    Helper function to search the STAC catalog with a specific cloud cover threshold.  
    Returns a list of found satellite scenes.
    '''
    bbox = get_bbox(row)
    
    search = catalog.search(
        collections=['landsat-c2-l2'],
        bbox=bbox,
        datetime='2011-01-01/2015-12-31',
        query={
            'eo:cloud_cover': {'lt': cloud_cover_query},
            'platform': {'in': ['landsat-5', 'landsat-7', 'landsat-8']}
        },
    )
    return search.item_collection()

def call_api(row):
    '''
    Queries the Planetary Computer STAC API with Connection Retry Logic.
    Prevents 502 Bad Gateway errors by retrying the connection up to 3 times.
    '''
    max_retries = 3
    base_delay = 2
    date = pd.to_datetime(row['Sample Date'], dayfirst=True, errors='coerce')

    # API Calls
    for attempt in range(max_retries):
        try:
            # Quality
            items = get_items(row, 10)
            if len(items) > 0: return items

            # More flexibility
            items = get_items(row, 30)
            if len(items) > 0: return items
            
            # Better than empty
            return get_items(row, 80)

        except (APIError, Exception) as e:
            if attempt == max_retries - 1:
                return [] 
            
            # Waits exponentially
            sleep_time = base_delay * (2 ** attempt)
            time.sleep(sleep_time)
            
    return []

In [ ]:
def process_bands(data):
    '''
    Computes the median value for each spectral band, handling zeros and NaNs.
    Returns a pandas Series with the extracted band values.
    '''
    results = {}

    for band in landsat_bands_of_interest:
        try:
            # Trys to convert to float
            if band in data:
                band_data = data[band].astype('float')
                
                # Compute median - ignores outliers (bords or dirty pixels)
                median_val = float(band_data.median(skipna=True).values)

                # Replace 0 with NaN
                median_val = median_val if median_val != 0 else np.nan

                # Adding to results
                results[band] = median_val
            else:
                results[band] = np.nan

        except Exception as e:
            results[band] = np.nan

    # Returns as pandas series
    return pd.Series(results)

In [2]:
def compute_Landsat_values(row):
    '''
    Main driver function:
    Retrieves available satellite scenes via API.
    Selects the scene closest to the sample date.
    Loads pixel data and extracts median band values.
    '''
    date = pd.to_datetime(row['Sample Date'], dayfirst=True, errors='coerce')
    bbox = get_bbox(row)
    items = call_api(row)
    
    if not items:
        print('No items to save')
        return error_row

    try:
        # Convert sample date to UTC
        sample_date_utc = date.tz_localize('UTC') if date.tzinfo is None else date.tz_convert('UTC')

        # Pick the item closest to the sample date
        items = sorted(
            items,
            key=lambda x: abs(pd.to_datetime(x.properties['datetime']).tz_convert('UTC') - sample_date_utc)
        )

        # Download pixels
        selected_item = pc.sign(items[0])

        # Load required bands
        data = stac_load(
            [selected_item],
            bands=landsat_bands_of_interest,
            bbox=bbox
        ).isel(time=0)
        
        return process_bands(data)
    
    except Exception as e:
        print(e)
        return error_row

In [ ]:
def parallel_extraction(df, max_workers=5):
    '''
    Helper function to execute feature extraction in parallel using threads.
    '''
    # Convert DataFrame to a list of dictionaries for faster iteration than native Pandas
    rows = df.to_dict('records')
    
    # 'executor.map' passes a raw dictionary, but 'compute_Landsat_values' expects a Pandas Series.
    def task_wrapper(row):
        result = compute_Landsat_values(pd.Series(row))
        result['Latitude'] = row['Latitude']
        result['Longitude'] = row['Longitude']
        result['Sample Date'] = row['Sample Date']
        return result

    results = []
    print(f'Starting parallel extraction with {max_workers} workers...')
    
    # Initialize the thread pool context manager
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # executor.map distributes the tasks across threads.
        # tqdm adds a progress bar.
        results = list(tqdm(
            executor.map(task_wrapper, rows), total=len(rows)
        ))
        
    return pd.DataFrame(results)

## Data transformation

In [ ]:
# Training
print('🚀 Running Landsat feature extraction for training data...')
landsat_train_features = parallel_extraction(water_quality_df)

print('')

# Testing
print('🚀 Running Landsat feature extraction for validation data...')
landsat_val_features = parallel_extraction(validation_df)

## Data saving

In [ ]:
save_df(landsat_train_features, 'landsat_train_features_base', save_path)
save_df(landsat_val_features, 'landsat_val_features_base', save_path)